# CacheBackedEmbeddings

Embeddings는 재계산을 피하기 위해 저장되거나 일시적으로 캐시될 수 있습니다.

Embeddings를 캐싱하는 것은 `CacheBackedEmbeddings`를 사용하여 수행될 수 있습니다. 캐시 지원 embedder는 embeddings를 키-값 저장소에 캐싱하는 embedder 주변에 래퍼입니다. 텍스트는 해시되고 이 해시는 캐시에서 키로 사용됩니다.

`CacheBackedEmbeddings`를 초기화하는 주요 지원 방법은 `from_bytes_store`입니다. 이는 다음 매개변수를 받습니다:

- `underlying_embeddings`: 임베딩을 위해 사용되는 embedder.
- `document_embedding_cache`: 문서 임베딩을 캐싱하기 위한 `ByteStore` 중 하나.
- `namespace`: (선택 사항, 기본값은 `""`) 문서 캐시를 위해 사용되는 네임스페이스. 이 네임스페이스는 다른 캐시와의 충돌을 피하기 위해 사용됩니다. 예를 들어, 사용된 임베딩 모델의 이름으로 설정하세요.

**주의**: 동일한 텍스트가 다른 임베딩 모델을 사용하여 임베딩될 때 충돌을 피하기 위해 `namespace` 매개변수를 설정하는 것이 중요합니다.


In [1]:
from dotenv import load_dotenv
load_dotenv()

True

## LocalFileStore 에서 임베딩 사용 (영구 보관)

먼저, 로컬 파일 시스템을 사용하여 임베딩을 저장하고 FAISS 벡터 스토어를 사용하여 검색하는 예제를 살펴보겠습니다.


In [1]:
from langchain_classic.storage import LocalFileStore
from langchain_openai import OpenAIEmbeddings
from langchain_classic.embeddings import CacheBackedEmbeddings
from langchain_community.vectorstores.faiss import FAISS

# OpenAI 임베딩을 사용하여 기본 임베딩 설정
embedding = OpenAIEmbeddings()

# 로컬 파일 저장소 설정
store = LocalFileStore("./cache/")

# 캐시를 지원하는 임베딩 생성
cached_embedder = CacheBackedEmbeddings.from_bytes_store(
    underlying_embeddings=embedding,
    document_embedding_cache=store,
    namespace=embedding.model,  # 기본 임베딩과 저장소를 사용하여 캐시 지원 임베딩을 생성
)

C:\Users\user\AppData\Local\Temp\ipykernel_29832\1030009795.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores.faiss import FAISS
d:\hykim\hanwha_0902\.venv\Lib\site-packages\langchain_classic\embeddings\cache.py:58: UserWarning: Using default key encoder: SHA-1 is *not* collision-resistant. While acceptable for most cache scenarios, a motivated attacker can craft two different payloads that map to the same cache key. If that risk matters in your environment, supply a stronger encoder (e.g. SHA-256 or BLAKE2) via the `key_encoder` argument. If you change the key encoder, consider also creating a new cache, to avoid (the potential for) collisions with existing keys.
  _warn_about_sha1_encoder()


In [2]:
# store에서 키들을 순차적으로 가져옵니다.
list(store.yield_keys())

[]

문서를 로드하고, 청크로 분할한 다음, 각 청크를 임베딩하고 벡터 저장소에 로드합니다.


In [ ]:
from langchain_classic.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter

# 문서 로드 ->  문자 인코딩 불일치 (교재 오류)
# raw_documents = TextLoader("./data/appendix-keywords.txt").load()

# 문서 로드
raw_documents = TextLoader(
    "./data/appendix-keywords.txt",
    encoding="utf-8",
).load()

# 문자 단위로 텍스트 분할 설정 -> 청크 사이즈를 너무 크게 잡음 (교재 오류)
text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=0)

# 문자 단위로 텍스트 분할 설정
text_splitter = CharacterTextSplitter(chunk_size=100, chunk_overlap=0)

documents = text_splitter.split_documents(raw_documents)

Created a chunk of size 169, which is longer than the specified 100
Created a chunk of size 162, which is longer than the specified 100
Created a chunk of size 140, which is longer than the specified 100
Created a chunk of size 170, which is longer than the specified 100
Created a chunk of size 142, which is longer than the specified 100
Created a chunk of size 201, which is longer than the specified 100
Created a chunk of size 202, which is longer than the specified 100
Created a chunk of size 181, which is longer than the specified 100
Created a chunk of size 175, which is longer than the specified 100
Created a chunk of size 203, which is longer than the specified 100
Created a chunk of size 177, which is longer than the specified 100
Created a chunk of size 167, which is longer than the specified 100
Created a chunk of size 201, which is longer than the specified 100
Created a chunk of size 148, which is longer than the specified 100
Created a chunk of size 176, which is longer tha

In [ ]:
# 코드 실행 시간을 측정합니다.
%time db = FAISS.from_documents(documents, cached_embedder)
# -> 문서로부터 FAISS 데이터베이스 생성

CPU times: total: 344 ms
Wall time: 3.77 s


벡터 저장소를 다시 생성하려고 하면, 임베딩을 다시 계산할 필요가 없기 때문에 훨씬 더 빠르게 처리됩니다.

In [10]:
# 캐싱된 임베딩을 사용하여 FAISS 데이터베이스 생성
%time db2 = FAISS.from_documents(documents, cached_embedder)

CPU times: total: 109 ms
Wall time: 666 ms


## `InmemoryByteStore` 사용 (비영구적)

다른 `ByteStore`를 사용하기 위해서는 `CacheBackedEmbeddings`를 생성할 때 해당 `ByteStore`를 사용하면 됩니다.

아래에서는, 비영구적인 `InMemoryByteStore`를 사용하여 동일한 캐시된 임베딩 객체를 생성하는 예시를 보여줍니다.


In [11]:
from langchain_classic.embeddings import CacheBackedEmbeddings
from langchain_classic.storage import InMemoryByteStore

# 메모리 내 바이트 저장소 생성
store = InMemoryByteStore() 

# 캐시 지원 임배딩 생성
cached_embedder = CacheBackedEmbeddings.from_bytes_store(
    embedding, store, namespace=embedding.model
)
